<a href="https://colab.research.google.com/github/Desire-in-tech/stock-market-volatility-forecasting/blob/main/notebooks/04_model_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — Model Deployment with FastAPI
**BSE Stock Market Volatility Forecasting**

**Goals:**
1. Understand the FastAPI server in `src/main.py`
2. Walk through the Pydantic data classes (`FitIn`, `FitOut`, `PredictIn`, `PredictOut`)
3. Start the server and hit `/hello`
4. Call `/fit` to train and save a GARCH model via the API
5. Call `/predict` to get a live volatility forecast
6. Explore the auto-generated interactive docs

## 1. Setup

In [4]:
import requests
import json
import pandas as pd

BASE_URL = "https://c529fae0-08d0-4473-9c3b-c85a622e38df-00-16c4wof8qu59z.spock.replit.dev"

print(f'Will connect to: {BASE_URL}')
print("Make sure the server is running before executing the cells below.")

Will connect to: https://c529fae0-08d0-4473-9c3b-c85a622e38df-00-16c4wof8qu59z.spock.replit.dev
Make sure the server is running before executing the cells below.


## 2. Understanding `main.py`

Here is a summary of what each part of `src/main.py` does:

| Component | Purpose |
|---|---|
| `FastAPI()` | Creates the app instance |
| `FitIn` | Pydantic model for `/fit` request body |
| `FitOut` | Pydantic model for `/fit` response (inherits `FitIn`) |
| `PredictIn` | Pydantic model for `/predict` request body |
| `PredictOut` | Pydantic model for `/predict` response (inherits `PredictIn`) |
| `GET /hello` | Health check |
| `POST /fit` | Fetch data → fit GARCH → save model → return metadata |
| `POST /predict` | Load latest model → forecast → return volatility dict |

Let's look at the key Pydantic models:

In [5]:
# ── Pydantic data classes (mirrors src/main.py) ──────────────────────────────
from pydantic import BaseModel
from typing import Optional

class FitIn(BaseModel):
    ticker: str
    start_date: str
    end_date: str
    n_observations: int = 2500
    p: int = 1
    q: int = 1

class FitOut(FitIn):
    success: bool
    message: str
    model_path: str
    aic: float
    bic: float

class PredictIn(BaseModel):
    ticker: str
    n_days: int = 5

class PredictOut(PredictIn):
    success: bool
    model_path: str
    forecast: dict

# ── Validate example payloads ──────────────────────────────────────────────
fit_in = FitIn(
    ticker='RELIANCE.NS',
    start_date='2018-01-01',
    end_date='2024-12-31',
    n_observations=1500,
    p=1, q=1
)
print('FitIn model (serialised):')
print(json.dumps(fit_in.model_dump(), indent=2))

predict_in = PredictIn(ticker='RELIANCE.NS', n_days=5)
print('\nPredictIn model:')
print(json.dumps(predict_in.model_dump(), indent=2))

FitIn model (serialised):
{
  "ticker": "RELIANCE.NS",
  "start_date": "2018-01-01",
  "end_date": "2024-12-31",
  "n_observations": 1500,
  "p": 1,
  "q": 1
}

PredictIn model:
{
  "ticker": "RELIANCE.NS",
  "n_days": 5
}


## 3. `/hello` — Health Check

A simple GET request to confirm the server is alive.

In [6]:
response = requests.get(f'{BASE_URL}/hello')

print(f'Status code : {response.status_code}')
print(f'Response    : {json.dumps(response.json(), indent=2)}')

Status code : 200
Response    : {
  "message": "Hello! BSE Volatility Forecasting API is live.",
  "timestamp": "2026-06-04T14:41:43.180652Z"
}


## 4. `/fit` — Train and Save a GARCH Model

This endpoint:
1. Downloads data from Yahoo Finance for the given `ticker`
2. Stores it in SQLite
3. Fits a GARCH(p, q) model
4. Saves the model to `models/<ticker>_<date>.pkl`
5. Returns the model's AIC, BIC, and file path

In [7]:
FIT_PAYLOAD = {
    'ticker': 'RELIANCE.NS',
    'start_date': '2018-01-01',
    'end_date': '2024-12-31',
    'n_observations': 1500,
    'p': 1,
    'q': 1
}

print('Sending /fit request (this may take ~15–30 seconds for data download + fitting)...')
response = requests.post(f'{BASE_URL}/fit', json=FIT_PAYLOAD)

print(f'\nStatus code : {response.status_code}')
result = response.json()
print(json.dumps(result, indent=2))

Sending /fit request (this may take ~15–30 seconds for data download + fitting)...

Status code : 200
{
  "ticker": "RELIANCE.NS",
  "start_date": "2018-01-01",
  "end_date": "2024-12-31",
  "n_observations": 1500,
  "p": 1,
  "q": 1,
  "success": true,
  "message": "Model fitted and saved for RELIANCE.NS.",
  "model_path": "/home/runner/workspace/stock-market-volatility-forecasting/models/RELIANCE.NS_2026-06-04.pkl",
  "aic": 5744.7481,
  "bic": 5765.9984
}


In [8]:
# Parse the FitOut response into a Pydantic model for type-safe access
if response.status_code == 200:
    fit_out = FitOut(**result)
    print(f'Success     : {fit_out.success}')
    print(f'Ticker      : {fit_out.ticker}')
    print(f'AIC         : {fit_out.aic}')
    print(f'BIC         : {fit_out.bic}')
    print(f'Model path  : {fit_out.model_path}')

Success     : True
Ticker      : RELIANCE.NS
AIC         : 5744.7481
BIC         : 5765.9984
Model path  : /home/runner/workspace/stock-market-volatility-forecasting/models/RELIANCE.NS_2026-06-04.pkl


## 5. Fit More Stocks

In [9]:
# Fit models for multiple BSE/NSE tickers
TICKERS_TO_FIT = ['TCS.NS', 'INFY.NS', '^BSESN']

fit_results = []
for ticker in TICKERS_TO_FIT:
    payload = {
        'ticker': ticker,
        'start_date': '2018-01-01',
        'end_date': '2024-12-31',
        'n_observations': 1500,
        'p': 1, 'q': 1
    }
    print(f'Fitting {ticker}...')
    r = requests.post(f'{BASE_URL}/fit', json=payload)
    data = r.json()
    if r.status_code == 200:
        fit_results.append({'ticker': ticker, 'aic': data['aic'], 'bic': data['bic'], 'success': data['success']})
        print(f'  OK — AIC={data["aic"]:.2f}, BIC={data["bic"]:.2f}')
    else:
        print(f'  ERROR: {data}')

print('\nAll models fitted:')
pd.DataFrame(fit_results)

Fitting TCS.NS...
  OK — AIC=5343.05, BIC=5364.30
Fitting INFY.NS...
  OK — AIC=5727.60, BIC=5748.85
Fitting ^BSESN...
  OK — AIC=4072.45, BIC=4093.70

All models fitted:


,ticker,aic,bic,success
0,TCS.NS,5343.0454,5364.2956,True
1,INFY.NS,5727.5959,5748.8461,True
2,^BSESN,4072.4547,4093.7049,True


## 6. `/predict` — Forecast Volatility

This endpoint:
1. Finds the most recent saved model for `ticker`
2. Loads it from disk
3. Produces an annualised volatility forecast for `n_days` trading days ahead
4. Returns the forecast as a dict (`h.1`, `h.2`, …, `h.n`)

In [10]:
PREDICT_PAYLOAD = {
    'ticker': 'RELIANCE.NS',
    'n_days': 5
}

response = requests.post(f'{BASE_URL}/predict', json=PREDICT_PAYLOAD)

print(f'Status code : {response.status_code}')
result = response.json()
print(json.dumps(result, indent=2))

Status code : 200
{
  "ticker": "RELIANCE.NS",
  "n_days": 5,
  "success": true,
  "model_path": "/home/runner/workspace/stock-market-volatility-forecasting/models/RELIANCE.NS_2026-06-04.pkl",
  "forecast": {
    "h.1": 21.644741,
    "h.2": 21.973155,
    "h.3": 22.283018,
    "h.4": 22.575671,
    "h.5": 22.852322
  }
}


In [11]:
if response.status_code == 200:
    predict_out = PredictOut(**result)
    print(f'Ticker: {predict_out.ticker}')
    print(f'Model : {predict_out.model_path}')
    print()
    print('Annualised Volatility Forecast:')
    for day_label, vol in predict_out.forecast.items():
        print(f'  {day_label} : {vol:.2f}%')

Ticker: RELIANCE.NS
Model : /home/runner/workspace/stock-market-volatility-forecasting/models/RELIANCE.NS_2026-06-04.pkl

Annualised Volatility Forecast:
  h.1 : 21.64%
  h.2 : 21.97%
  h.3 : 22.28%
  h.4 : 22.58%
  h.5 : 22.85%


## 7. Compare Forecasts Across Stocks

In [12]:
import plotly.graph_objects as go

tickers_to_predict = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', '^BSESN']
forecast_data = {}

for ticker in tickers_to_predict:
    r = requests.post(f'{BASE_URL}/predict', json={'ticker': ticker, 'n_days': 5})
    if r.status_code == 200:
        forecast_data[ticker] = r.json()['forecast']
    else:
        print(f'Prediction failed for {ticker}: {r.json()}')

# Plot
fig = go.Figure()
days = list(range(1, 6))

for ticker, forecast in forecast_data.items():
    vols = list(forecast.values())
    fig.add_trace(go.Scatter(
        x=[f'Day {d}' for d in days],
        y=vols,
        mode='lines+markers',
        name=ticker
    ))

fig.update_layout(
    title='5-Day GARCH Volatility Forecast — BSE/NSE Stocks',
    xaxis_title='Forecast Horizon',
    yaxis_title='Annualised Volatility (%)',
    template='plotly_white',
    height=430
)
fig.show()

## 8. Error Handling

The API returns informative errors when things go wrong.

In [13]:
# Try predicting for a ticker that hasn't been fitted yet
r = requests.post(f'{BASE_URL}/predict', json={'ticker': 'NOTFITTED.NS', 'n_days': 5})
print(f'Status code : {r.status_code}  (expected 404)')
print(f'Detail      : {r.json()["detail"]}')

Status code : 404  (expected 404)
Detail      : No saved model found for ticker 'NOTFITTED.NS'. Call /fit first.


In [14]:
# Try fitting an invalid ticker
r = requests.post(f'{BASE_URL}/fit', json={
    'ticker': 'INVALID_TICKER_XYZ',
    'start_date': '2023-01-01',
    'end_date': '2024-01-01',
})
print(f'Status code : {r.status_code}  (expected 400)')
print(f'Detail      : {r.json()["detail"]}')

Status code : 400  (expected 400)
Detail      : No data returned for ticker 'INVALID_TICKER_XYZ'. Check the symbol and date range.


## 9. Interactive API Docs

FastAPI auto-generates two documentation UIs:

When eunning locally:

- **Swagger UI**: http://localhost:8008/docs — try endpoints interactively
- **ReDoc**: http://localhost:8008/redoc — clean reference documentation

When deployed:
- **Swagger UI**: "https://c529fae0-08d0-4473-9c3b-c85a622e38df-00-16c4wof8qu59z.spock.replit.dev"/docs
- **ReDoc**: "https://c529fae0-08d0-4473-9c3b-c85a622e38df-00-16c4wof8qu59z.spock.replit.dev"/redoc

Open these in your browser while the server is running.

In [15]:
# Fetch and display the OpenAPI schema
schema = requests.get(f'{BASE_URL}/openapi.json').json()
print('API title    :', schema['info']['title'])
print('API version  :', schema['info']['version'])
print('Paths        :', list(schema['paths'].keys()))

API title    : BSE Volatility Forecasting API
API version  : 1.0.0
Paths        : ['/hello', '/fit', '/predict']


## Summary

We have built a complete end-to-end volatility forecasting system:

| Step | Tool | Output |
|---|---|---|
| Data collection | `yfinance` | Clean DataFrame with returns |
| Data storage | `SQLite` + `SQLAlchemy` | Persistent stock price DB |
| Volatility modelling | `arch` GARCH(1,1) | Conditional volatility |
| Model persistence | `joblib` | `.pkl` model checkpoints |
| API deployment | `FastAPI` + `uvicorn` | Live prediction endpoint |

**The API is production-ready:**
- Input validation via Pydantic
- Structured error responses
- Auto-generated OpenAPI docs
- Stateless `POST /predict` — just send the ticker, get the forecast